<!--
---
PURPOSE: Run zero-label pose estimation — Facemap (eye pupil + face SVD) and
         SuperAnimal-quadruped (side body keypoints). Outputs saved to
         outputs/pose/ for use in NB07 and NB08.
REQUIRES:
  - data/raw/visual-behavior-neuropixels/{session_id}/behavior_videos/eye.mp4
  - data/raw/visual-behavior-neuropixels/{session_id}/behavior_videos/face.mp4
  - data/raw/visual-behavior-neuropixels/{session_id}/behavior_videos/side.mp4
PRODUCES:
  - outputs/pose/session_{id}_facemap_eye.parquet   (pupil area/pos/blink)
  - outputs/pose/session_{id}_facemap_face.parquet  (SVD motion energy PCs)
  - outputs/pose/session_{id}_superanimal.h5        (body keypoints)
WHAT_NEXT: notebooks/07_Pose_to_Motifs_Feature_Engineering.ipynb
---
-->

# 06 Pose Estimation — Facemap + SuperAnimal

**Cameras and tools:**
| Camera | Tool | Labels needed | Output |
|--------|------|---------------|--------|
| `eye.mp4` | Facemap pupil tracker | 0 | pupil area, x/y position, blink |
| `face.mp4` | Facemap SVD motion energy | 0 | top-N motion energy PCs |
| `side.mp4` | SuperAnimal-quadruped (DLC) | 0 | body keypoints (paw, spine, tail) |

**Install once (in vbn-analysis env):**
```bash
pip install facemap
pip install "deeplabcut[tf]"
```


## Environment Setup
We add the repo `src/` to the Python path so notebooks can import shared modules.

In [7]:
import sys
import importlib
import inspect
from pathlib import Path

# Resolve repo root whether notebook was launched from repo root or notebooks/.
ROOT = Path.cwd().resolve()
if (ROOT / "src").exists() and (ROOT / "notebooks").exists():
    pass
elif (ROOT.parent / "src").exists() and (ROOT.parent / "notebooks").exists():
    ROOT = ROOT.parent
else:
    raise RuntimeError(f"Could not resolve project root from cwd={Path.cwd()}")

SRC = ROOT / "src"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Ensure notebook uses the local io_sessions module from this repo.
io_sessions = importlib.import_module("io_sessions")
io_sessions = importlib.reload(io_sessions)
print("io_sessions:", io_sessions.__file__)
print("get_session_bundle:", inspect.signature(io_sessions.get_session_bundle))


io_sessions: /Users/muh/projects/vbn-analysis/src/io_sessions.py
get_session_bundle: (session_id: 'int', sessions_df: 'pd.DataFrame | None' = None, *, resolve_nwb: 'bool' = True, inspect_modalities: 'bool' = True) -> 'SessionBundle'


## Prerequisite Check
We parse the notebook header and validate required artifacts before running downstream steps.

In [11]:
from pathlib import Path
from reports import parse_notebook_header, validate_prerequisites

nb_path = ROOT / "notebooks" / "06_Pose_Estimation_Setup_SLEAP_or_DLC.ipynb"
header  = parse_notebook_header(nb_path)
missing = validate_prerequisites(header.get("REQUIRES", []))

if missing:
    print("Missing prerequisites:")
    for item in missing:
        print(" -", item)
else:
    print("All prerequisites satisfied.")

All prerequisites satisfied.


/opt/anaconda3/envs/vbn-analysis/lib/python3.10/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


## Step 1: Configuration
Set session IDs and number of SVD components to keep from face.mp4.


In [13]:
from config import get_config
from io_sessions import load_sessions_csv, get_session_bundle
import pandas as pd
import numpy as np
from pathlib import Path

cfg = get_config()
sessions = load_sessions_csv()
SESSION_IDS = [1055240613]

N_SVD_COMPONENTS = 20   # face.mp4 motion energy PCs to keep
CAMERAS = ["eye", "face", "side"]

VIDEO_ROOT = cfg.video_cache_dir
POSE_OUT   = cfg.outputs_dir / "pose"
POSE_OUT.mkdir(parents=True, exist_ok=True)

def video_path(session_id, camera):
    return VIDEO_ROOT / str(session_id) / "behavior_videos" / f"{camera}.mp4"

# Check what videos are available
for sid in SESSION_IDS:
    for cam in CAMERAS:
        p = video_path(sid, cam)
        status = f"{p.stat().st_size/1024**3:.2f} GB" if p.exists() else "MISSING"
        print(f"  [{sid}] {cam}.mp4: {status}")


In [15]:
# ── Facemap: eye pupil tracking + face SVD ─────────────────────────────────
# Requires: pip install facemap
# eye.mp4  → pupil area, x/y position, blink mask (ellipse fit)
# face.mp4 → SVD motion energy PCs (captures whisking, sniffing, grooming)

try:
    import facemap
    from facemap import process
    _facemap_available = True
except ImportError:
    _facemap_available = False
    print("Facemap not installed. Run: pip install facemap")

from timebase import write_parquet_with_timebase
from config import make_provenance

if _facemap_available:
    for sid in SESSION_IDS:
        # ── eye.mp4: pupil tracking ──────────────────────────────────────────
        eye_vid = video_path(sid, "eye")
        eye_out = POSE_OUT / f"session_{sid}_facemap_eye.parquet"

        if eye_out.exists():
            print(f"[{sid}] eye facemap already cached — skipping")
        elif not eye_vid.exists():
            print(f"[{sid}] eye.mp4 not found — download via NB01 cell 11")
        else:
            print(f"[{sid}] Running Facemap pupil tracking on eye.mp4 ...")
            try:
                # Facemap process() returns proc dict with pupil, blink, etc.
                proc = process.run(
                    filenames=[[str(eye_vid)]],
                    sbin=4,           # spatial binning (px)
                    proc=None,
                    savepath=str(POSE_OUT),
                    do_mot_svd=False, # no SVD on eye (use pupil mode)
                    do_pupil=True,
                    do_blink=True,
                    do_running=False,
                )
                # Extract pupil timeseries
                n_frames = len(proc.get("pupil", [{}])[0].get("area", []))
                # Build timestamps: use frame index / fps as placeholder
                # (aligned to NWB timebase in NB07 via frame_times)
                fps = proc.get("Ly", [30])[0] if "fps" not in proc else proc["fps"]
                t_approx = np.arange(n_frames) / 30.0  # replaced in NB07
                pupil_data = proc["pupil"][0]
                df_eye = pd.DataFrame({
                    "frame": np.arange(n_frames),
                    "t": t_approx,
                    "pupil_area": pupil_data.get("area", np.full(n_frames, np.nan)),
                    "pupil_x":    pupil_data.get("com", np.full((n_frames,2), np.nan))[:,0] if "com" in pupil_data else np.nan,
                    "pupil_y":    pupil_data.get("com", np.full((n_frames,2), np.nan))[:,1] if "com" in pupil_data else np.nan,
                })
                if "blink" in proc and proc["blink"]:
                    df_eye["blink"] = proc["blink"][0].astype(bool)
                write_parquet_with_timebase(
                    df_eye, eye_out,
                    provenance=make_provenance(sid, "facemap_eye"),
                    required_columns=["frame"],
                )
                print(f"  Saved: {eye_out} ({len(df_eye)} frames)")
            except Exception as e:
                print(f"  ERROR running Facemap on eye.mp4: {e}")

        # ── face.mp4: SVD motion energy ──────────────────────────────────────
        face_vid = video_path(sid, "face")
        face_out = POSE_OUT / f"session_{sid}_facemap_face.parquet"

        if face_out.exists():
            print(f"[{sid}] face SVD already cached — skipping")
        elif not face_vid.exists():
            print(f"[{sid}] face.mp4 not found — download via NB01 cell 11")
        else:
            print(f"[{sid}] Running Facemap SVD on face.mp4 ...")
            try:
                proc = process.run(
                    filenames=[[str(face_vid)]],
                    sbin=4,
                    proc=None,
                    savepath=str(POSE_OUT),
                    do_mot_svd=True,
                    do_pupil=False,
                    do_blink=False,
                    do_running=False,
                )
                # mot_svd shape: (n_frames, n_components)
                mot_svd = proc.get("motSVD", [None])[0]
                if mot_svd is not None:
                    n_frames = mot_svd.shape[0]
                    n_keep = min(N_SVD_COMPONENTS, mot_svd.shape[1])
                    t_approx = np.arange(n_frames) / 30.0
                    svd_cols = {f"face_svd_{i}": mot_svd[:, i] for i in range(n_keep)}
                    df_face = pd.DataFrame({"frame": np.arange(n_frames), "t": t_approx, **svd_cols})
                    write_parquet_with_timebase(
                        df_face, face_out,
                        provenance=make_provenance(sid, "facemap_face_svd"),
                        required_columns=["frame"],
                    )
                    print(f"  Saved: {face_out} ({n_frames} frames, {n_keep} SVD components)")
                else:
                    print("  WARNING: motSVD not in facemap output")
            except Exception as e:
                print(f"  ERROR running Facemap on face.mp4: {e}")


Labeling outputs: /Users/muh/projects/vbn-analysis/outputs/labeling/sleap/1043752325/eye
Labeling outputs: /Users/muh/projects/vbn-analysis/outputs/labeling/sleap/1043752325/face
Labeling outputs: /Users/muh/projects/vbn-analysis/outputs/labeling/sleap/1043752325/side
Pose project: /Users/muh/projects/vbn-analysis/pose_projects/session_1043752325/sleap


In [ ]:
# ── SuperAnimal-quadruped: body keypoints from side.mp4 ─────────────────────
# Requires: pip install "deeplabcut[tf]"
# Uses superanimal_quadruped pretrained model — zero labels needed.
# Outputs: HDF5 with x/y/likelihood per keypoint per frame.

try:
    import deeplabcut as dlc
    _dlc_available = True
except ImportError:
    _dlc_available = False
    print("DeepLabCut not installed. Run: pip install \"deeplabcut[tf]\"")

if _dlc_available:
    import tempfile, shutil

    for sid in SESSION_IDS:
        side_vid = video_path(sid, "side")
        sa_out   = POSE_OUT / f"session_{sid}_superanimal.h5"

        if sa_out.exists():
            print(f"[{sid}] SuperAnimal already cached — skipping")
            continue
        if not side_vid.exists():
            print(f"[{sid}] side.mp4 not found — download via NB01 cell 11")
            continue

        print(f"[{sid}] Running SuperAnimal-quadruped on side.mp4 ...")
        print("  (First run downloads the pretrained model weights ~500 MB)")
        try:
            # SuperAnimal video inference — no project config needed
            dlc.video_inference_superanimal(
                videos=[str(side_vid)],
                superanimal_name="superanimal_quadruped",
                scale_list=[200],   # resize shorter dim to 200px for speed
                videotype="mp4",
            )
            # DLC saves results next to the video — move to outputs/pose/
            expected = side_vid.with_suffix("") 
            # find the generated h5
            h5_candidates = list(side_vid.parent.glob(f"{side_vid.stem}*superanimal*.h5"))
            if h5_candidates:
                shutil.move(str(h5_candidates[0]), str(sa_out))
                print(f"  Saved: {sa_out}")
            else:
                print("  WARNING: Could not find DLC output .h5 file")
        except Exception as e:
            print(f"  ERROR running SuperAnimal: {e}")
            print("  Try: dlc.video_inference_superanimal(videos=[str(side_vid)], superanimal_name=\"superanimal_quadruped\")")

print("\nNB06 complete. Run NB07 to convert pose outputs to feature timeseries.")
